<h1>Pair time-walk correction</h1><p>接续 5.0 的 event building，从同一 cluster 内的高能 pair 求相对 offset，再合并有效参考通道，拟合各晶体的 time walk。先看 cluster 6 的图和参数，再处理所有 cluster。下面保留完整实现，也可使用同目录 <a href="pair_timewalk_correction.C">macro</a>。</p>

In [1]:
%jsroot on

In [2]:
%%cpp -d
// pair_timewalk_correction.C

#include <iostream>
#include <vector>
#include <cmath>

#include "TFile.h"
#include "TTree.h"
#include "TH1F.h"
#include "TH2F.h"
#include "TProfile.h"
#include "TF1.h"
#include "TCanvas.h"
#include "TROOT.h"
#include "TStyle.h"
#include "TLatex.h"
#include "TFitResultPtr.h"

const int MAXHIT = 1024;
const int NCLUSTER = 12;
const int NSEG = 7;

double E_HIGH = 600.0;
double E_REF = 600.0;
double E_MAX_OFFSET = 3000.0;

TH2F *h_tw_id[NSEG] = {0};
TProfile *hp_tw_id[NSEG] = {0};

TH1F *h_off_pair[NSEG][NSEG] = {0};
TF1  *f_off_pair[NSEG][NSEG] = {0};

double t_off[NCLUSTER][NSEG][NSEG];
int has_toff[NCLUSTER][NSEG][NSEG];

TF1 *f_tw[NCLUSTER][NSEG] = {0};
int has_fit[NCLUSTER][NSEG];

Long64_t seg_high_count[NCLUSTER][NSEG];

double seg_offset[NCLUSTER][NSEG];
int has_seg_offset[NCLUSTER][NSEG];

int ref_seg_cluster[NCLUSTER];

// Make pair-combination time-walk histograms for one cluster

先用高能晶体对估计固定 offset，再逐事件扣除 offset，填入各晶体的 time-walk 图。两次循环分别回答两个问题，使用的能量条件不同。

In [3]:
%%cpp -d
std::vector<TH2F*> make_pair_timewalk_cluster(int clusterID = 6)
{
  TH1::AddDirectory(kFALSE);

  std::vector<TH2F*> hlist;
  if (clusterID < 0 || clusterID >= NCLUSTER) {
    std::cout << "invalid clusterID" << std::endl;
    return hlist;
  }

  TFile *fin = new TFile("eurica_event.root");
  if (!fin || fin->IsZombie()) {
    std::cout << "cannot open eurica_event.root" << std::endl;
    return hlist;
  }

  TTree *tree = (TTree*)fin->Get("tree");
  if (!tree) {
    std::cout << "cannot find tree" << std::endl;
    fin->Close();
    return hlist;
  }

  int ghit;
  int gid[MAXHIT];
  double ge[MAXHIT];
  double gt[MAXHIT];
  if (tree->GetMaximum("ghit")>MAXHIT)
    throw std::runtime_error("Increase MAXHIT before reading branches");

  tree->SetBranchAddress("ghit", &ghit);
  tree->SetBranchAddress("gid", gid);
  tree->SetBranchAddress("ge", ge);
  tree->SetBranchAddress("gt", gt);
  for (int s = 0; s < NSEG; s++) {
    seg_high_count[clusterID][s] = 0;
  }

  // reset offset histograms and fit functions
  for (int i = 0; i < NSEG; i++) {
    for (int j = 0; j < NSEG; j++) {

      has_toff[clusterID][i][j] = 0;
      t_off[clusterID][i][j] = 0.0;
      if (h_off_pair[i][j]) {
        delete h_off_pair[i][j];
        h_off_pair[i][j] = 0;
      }
      if (f_off_pair[i][j]) {
        delete f_off_pair[i][j];
        f_off_pair[i][j] = 0;
      }
      if (i == j) continue;

      h_off_pair[i][j] =
        new TH1F(Form("h_off_c%d_%d_%d", clusterID, i, j),
                 Form("cluster %d: pair %d-%d; t_{%d}-t_{%d} (ns); counts",
                      clusterID, i, j, i, j),
                 120, -600, 600);

      h_off_pair[i][j]->SetDirectory(0);
    }
  }

  Long64_t nentries = tree->GetEntries();

  // first pass: fill high-energy pair offset histograms
  for (Long64_t ientry = 0; ientry < nentries; ientry++) {

    tree->GetEntry(ientry);
    for (int a = 0; a < ghit; a++) {

      int cid_a = gid[a] / 7;
      int sid_a = gid[a] % 7;
      if (cid_a != clusterID) continue;
      if (sid_a < 0 || sid_a >= NSEG) continue;
      if (ge[a] < E_HIGH || ge[a] > E_MAX_OFFSET) continue;

      seg_high_count[clusterID][sid_a]++;
      for (int b = 0; b < ghit; b++) {
        if (a == b) continue;

        int cid_b = gid[b] / 7;
        int sid_b = gid[b] % 7;
        if (cid_b != clusterID) continue;
        if (sid_b < 0 || sid_b >= NSEG) continue;
        if (sid_b == sid_a) continue;
        if (ge[b] < E_HIGH || ge[b] > E_MAX_OFFSET) continue;

        double dt = gt[a] - gt[b];

        h_off_pair[sid_a][sid_b]->Fill(dt);
      }
    }
  }

  // fit pair offsets
  for (int i = 0; i < NSEG; i++) {
    for (int j = 0; j < NSEG; j++) {
      if (i == j) continue;
      if (!h_off_pair[i][j]) continue;
      if (h_off_pair[i][j]->GetEntries() < 50) {
        continue;
      }

      int maxbin = h_off_pair[i][j]->GetMaximumBin();
      double x0 = h_off_pair[i][j]->GetBinCenter(maxbin);

      f_off_pair[i][j] =
        new TF1(Form("f_off_c%d_%d_%d", clusterID, i, j),
                "gaus",
                x0 - 120.0,
                x0 + 120.0);

      f_off_pair[i][j]->SetParameter(0, h_off_pair[i][j]->GetMaximum());
      f_off_pair[i][j]->SetParameter(1, x0);
      f_off_pair[i][j]->SetParameter(2, 60.0);

      TFitResultPtr r = h_off_pair[i][j]->Fit(f_off_pair[i][j], "RQ0");
      if ((int)r == 0) {
        t_off[clusterID][i][j] = f_off_pair[i][j]->GetParameter(1);
      } else {
        std::cout << "Skip failed pair offset fit: " << clusterID << ", " << i << ", " << j << std::endl;
        continue;
      }

      has_toff[clusterID][i][j] = 1;
    }
  }

  // reset time-walk histograms
  for (int s = 0; s < NSEG; s++) {
    if (h_tw_id[s]) {
      delete h_tw_id[s];
      h_tw_id[s] = 0;
    }

    h_tw_id[s] =
      new TH2F(Form("h_tw_c%d_id%d", clusterID, s),
               Form("cluster %d: segment %d, all references; #Delta t_{ij} (ns); E_{seg %d}",
                    clusterID, s, s),
               80, -600, 600,
               130, 0, 1300);

    h_tw_id[s]->SetDirectory(0);
    hlist.push_back(h_tw_id[s]);
  }

  // second pass: fill pair-offset-corrected time-walk plots
  for (Long64_t ientry = 0; ientry < nentries; ientry++) {

    tree->GetEntry(ientry);
    for (int a = 0; a < ghit; a++) {

      int cid_a = gid[a] / 7;
      int sid_a = gid[a] % 7;
      if (cid_a != clusterID) continue;
      if (sid_a < 0 || sid_a >= NSEG) continue;
      if (ge[a] < 30) continue;
      for (int b = 0; b < ghit; b++) {
        if (a == b) continue;

        int cid_b = gid[b] / 7;
        int sid_b = gid[b] % 7;
        if (cid_b != clusterID) continue;
        if (sid_b < 0 || sid_b >= NSEG) continue;
        if (sid_b == sid_a) continue;
        if (ge[b] < E_HIGH) continue;
        if (!has_toff[clusterID][sid_a][sid_b]) continue;

        double dt = gt[a] - gt[b];
        double dt_corr = dt - t_off[clusterID][sid_a][sid_b];

        h_tw_id[sid_a]->Fill(dt_corr, ge[a]);
      }
    }
  }

  fin->Close();

  std::cout << "cluster " << clusterID
            << " pair time-walk histograms filled." << std::endl;
  for (int s = 0; s < NSEG; s++) {
    std::cout << "segment " << s
              << " high-energy count = " << seg_high_count[clusterID][s]
              << ", walk entries = " << h_tw_id[s]->GetEntries()
              << std::endl;
  }

  return hlist;
}

In [4]:
%%cpp -d
void draw_pair_timewalk_cluster(int clusterID = 6)
{
  TCanvas *c_old = (TCanvas*)gROOT->FindObject("c_pair_tw");
  if (c_old) delete c_old;

  TCanvas *c = new TCanvas("c_pair_tw",
                           Form("cluster %d: pair-combination time walk", clusterID),
                           900, 600);

  c->Divide(4, 2);
  gStyle->SetOptStat(0);
  for (int s = 0; s < NSEG; s++) {
    c->cd(s + 1);
    if (h_tw_id[s]) h_tw_id[s]->Draw("colz");
  }

  c->Draw();
}

In [5]:
%%cpp -d
void draw_pair_offset_cluster(int clusterID = 6)
{
  TCanvas *c_old = (TCanvas*)gROOT->FindObject("c_pair_off");
  if (c_old) delete c_old;

  TCanvas *c = new TCanvas("c_pair_off",
                           Form("cluster %d: pair offsets", clusterID),
                           1050, 1050);

  c->Divide(7, 7);
  gStyle->SetOptStat(0);
  for (int i = 0; i < NSEG; i++) {
    for (int j = 0; j < NSEG; j++) {

      int ipad = i * NSEG + j + 1;
      c->cd(ipad);
      if (i == j) {
        TLatex text;
        text.SetTextAlign(22);
        text.SetTextSize(0.18);
        text.DrawLatexNDC(0.5, 0.5, Form("%d = %d", i, j));
        continue;
      }
      if (!h_off_pair[i][j]) continue;

      h_off_pair[i][j]->SetLineColor(kBlack);
      h_off_pair[i][j]->Draw("hist");
      if (f_off_pair[i][j]) {
        f_off_pair[i][j]->SetLineColor(kRed);
        f_off_pair[i][j]->Draw("same");
      }

      TLatex label;
      label.SetTextSize(0.10);
      label.SetNDC();
      if (has_toff[clusterID][i][j]) {
        label.DrawLatex(0.18, 0.82,
                        Form("%d-%d: %.1f ns", i, j,
                             t_off[clusterID][i][j]));
      } else {
        label.DrawLatex(0.18, 0.82,
                        Form("%d-%d: no fit", i, j));
      }
    }
  }

  c->Draw();
}

在有有效 time-walk 拟合的通道中，选取高能计数最多的一条作为参考。

In [6]:
%%cpp -d
int ChooseReferenceSegmentByHighCount(int clusterID)
{
  int bestSeg = -1;
  Long64_t bestCount = -1;
  for (int s = 0; s < NSEG; s++) {
    if (!has_fit[clusterID][s]) continue;

    Long64_t count = seg_high_count[clusterID][s];

    std::cout << "cluster " << clusterID
              << ", reference candidate segment " << s
              << ": high-energy count = "
              << count << std::endl;
    if (count > bestCount) {
      bestCount = count;
      bestSeg = s;
    }
  }

  return bestSeg;
}

把晶体间的固定 offset 并入参数 p0；优先采用直接参考，缺少直接 pair 时才沿有效参考关系传递。

In [7]:
%%cpp -d
void absorb_segment_offset_to_p0(int clusterID)
{
  if (clusterID < 0 || clusterID >= NCLUSTER) return;

  int refSeg = ChooseReferenceSegmentByHighCount(clusterID);
  ref_seg_cluster[clusterID] = refSeg;
  if (refSeg < 0) {
    std::cout << "cluster " << clusterID
              << ": no valid reference segment found." << std::endl;
    return;
  }

  std::cout << "cluster " << clusterID
            << ": selected reference segment = "
            << refSeg << std::endl;

  // C_s = (t_s-t_j) + C_j：无直接 pair 时，沿有效参考关系传递。
  for (int s=0; s<NSEG; ++s) {
    seg_offset[clusterID][s]=0;
    has_seg_offset[clusterID][s]=(s==refSeg);
    // 优先保留直接参考；仅对缺少直接 pair 的通道使用传递。
    if (s!=refSeg && has_toff[clusterID][s][refSeg]) {
      seg_offset[clusterID][s]=t_off[clusterID][s][refSeg];
      has_seg_offset[clusterID][s]=1;
    }
  }
  for (int step=0; step<NSEG; ++step)
    for (int s=0; s<NSEG; ++s) {
      if (has_seg_offset[clusterID][s]) continue;
      for (int j=0; j<NSEG; ++j) {
        if (!has_seg_offset[clusterID][j] || !has_toff[clusterID][s][j]) continue;
        seg_offset[clusterID][s]=t_off[clusterID][s][j]+seg_offset[clusterID][j];
        has_seg_offset[clusterID][s]=1;
        break;
      }
    }
  for (int s = 0; s < NSEG; s++) {
    if (!has_fit[clusterID][s] || !f_tw[clusterID][s]) continue;
    double C = seg_offset[clusterID][s];
    if (!has_seg_offset[clusterID][s]) {
      std::cout << "cluster " << clusterID
                << ", segment " << s
                << ": no connected offset path to reference segment "
                << refSeg
                << ", keep this channel uncorrected" << std::endl;
      has_fit[clusterID][s] = 0;
      continue;
    }

    seg_offset[clusterID][s] = C;

    // Shift p0 so that f(E_REF) = C.
    double fref = f_tw[clusterID][s]->Eval(E_REF);
    double p0_old = f_tw[clusterID][s]->GetParameter(0);
    double p0_new = p0_old - fref + C;

    f_tw[clusterID][s]->SetParameter(0, p0_new);

    std::cout << "cluster " << clusterID
              << ", segment " << s
              << ": C = " << C
              << " ns, p0 -> " << p0_new
              << std::endl;
  }
}

ProfileY 给出各能量 bin 的平均时间差。先拟合能量依赖，再调整 p0。`R0Q` 分别指定拟合范围、不自动画线、减少屏幕输出；仍检查拟合状态。

In [8]:
%%cpp -d
void fit_pair_timewalk_cluster(int clusterID = 6, bool draw = true)
{
  if (clusterID < 0 || clusterID >= NCLUSTER) {
    std::cout << "invalid clusterID" << std::endl;
    return;
  }

  double fitEmin = 30.0;
  double fitEmax = 800.0;

  double dtMin = -400.0;
  double dtMax = 400.0;

  TCanvas *c = 0;
  if (draw) {
    TCanvas *c_old = (TCanvas*)gROOT->FindObject("c_fit_tw");
    if (c_old) delete c_old;

    c = new TCanvas("c_fit_tw",
                    Form("cluster %d time-walk fits", clusterID),
                    900, 600);

    c->Divide(4, 2);
    c->SetLogy(0);
  }
  for (int s = 0; s < NSEG; s++) {

    has_fit[clusterID][s] = 0;
    if (!h_tw_id[s]) continue;
    if (h_tw_id[s]->GetEntries() < 100) continue;

    int xbin1 = h_tw_id[s]->GetXaxis()->FindBin(dtMin);
    int xbin2 = h_tw_id[s]->GetXaxis()->FindBin(dtMax);
    if (hp_tw_id[s]) {
      delete hp_tw_id[s];
      hp_tw_id[s] = 0;
    }

    hp_tw_id[s] = h_tw_id[s]->ProfileY(Form("hp_tw_c%d_s%d", clusterID, s),
                                       xbin1, xbin2);

    hp_tw_id[s]->SetDirectory(0);
    if (f_tw[clusterID][s]) {
      delete f_tw[clusterID][s];
      f_tw[clusterID][s] = 0;
    }

    f_tw[clusterID][s] =
      new TF1(Form("f_tw_c%d_s%d", clusterID, s),
              "[0]+[1]/sqrt(x)+[2]/x+[3]/(x*x)",
              30.0, 1300.0);

    f_tw[clusterID][s]->SetParameters(0.0, 1000.0, -10000.0, 100000.0);

    int status = hp_tw_id[s]->Fit(f_tw[clusterID][s], "R0Q", "", fitEmin, fitEmax);
    if (status != 0) {
      std::cout << "Skip failed time-walk fit: " << clusterID << ", " << s << std::endl;
      continue;
    }

    has_fit[clusterID][s] = 1;

    std::cout << "cluster " << clusterID
              << ", segment " << s
              << " fit:"
              << " p0=" << f_tw[clusterID][s]->GetParameter(0)
              << ", p1=" << f_tw[clusterID][s]->GetParameter(1)
              << ", p2=" << f_tw[clusterID][s]->GetParameter(2)
              << ", p3=" << f_tw[clusterID][s]->GetParameter(3)
              << std::endl;
    if (draw) {
      c->cd(s + 1);
      gPad->SetLogy(0);

      hp_tw_id[s]->SetMarkerStyle(20);
      hp_tw_id[s]->SetMarkerSize(0.7);
      hp_tw_id[s]->SetMinimum(-300);
      hp_tw_id[s]->SetMaximum(300);

      hp_tw_id[s]->Draw();

      TF1 *f_draw = (TF1*)f_tw[clusterID][s]->Clone(
        Form("f_draw_c%d_s%d", clusterID, s)
      );

      f_draw->SetLineColor(kRed);
      f_draw->SetRange(30, 1300);
      f_draw->Draw("same");
    }
  }

  // After fitting walk shape, absorb high-energy segment offset into p0.
  absorb_segment_offset_to_p0(clusterID);
  if (draw) c->Draw();
}

In [9]:
%%cpp -d
void fit_all_clusters_pair()
{
  for (int c = 0; c < NCLUSTER; c++) {

    std::cout << "======================================" << std::endl;
    std::cout << "Processing cluster " << c << std::endl;

    make_pair_timewalk_cluster(c);
    fit_pair_timewalk_cluster(c, false);
  }

  std::cout << "All cluster time-walk fits finished." << std::endl;
}

输入晶体编号和能量，返回需要减去的时间修正。没有有效拟合的通道返回 0，后面同时写入状态标记。

In [10]:
%%cpp -d
double GetTimeWalkCorrection(int clusterID, int segID, double E)
{
  if (clusterID < 0 || clusterID >= NCLUSTER) return 0.0;
  if (segID < 0 || segID >= NSEG) return 0.0;
  if (!has_fit[clusterID][segID]) return 0.0;
  if (!f_tw[clusterID][segID]) return 0.0;
  if (E <= 0) return 0.0;

  return f_tw[clusterID][segID]->Eval(E);
}

逐 hit 修正时间，保留原来的能量、编号和原始时间。`tw_ok` 标记本次是否应用了有效修正。

In [11]:
%%cpp -d
void make_time_corrected_tree_pair(
    const char *inputFile = "eurica_event.root",
    const char *outputFile = "eurica_time_pair.root")
{
  TFile *fin = new TFile(inputFile);
  if (!fin || fin->IsZombie()) {
    std::cout << "cannot open " << inputFile << std::endl;
    return;
  }

  TTree *tin = (TTree*)fin->Get("tree");
  if (!tin) {
    std::cout << "cannot find tree in " << inputFile << std::endl;
    fin->Close();
    return;
  }

  int ghit;
  int gid_in[MAXHIT];
  double ge_in[MAXHIT];
  double gt_in[MAXHIT];
  if (tin->GetMaximum("ghit")>MAXHIT)
    throw std::runtime_error("Increase MAXHIT before reading branches");

  tin->SetBranchAddress("ghit", &ghit);
  tin->SetBranchAddress("gid", gid_in);
  tin->SetBranchAddress("ge", ge_in);
  tin->SetBranchAddress("gt", gt_in);

  TFile *fout = new TFile(outputFile, "recreate");
  TTree *tout = new TTree("tree", "pair time-walk corrected tree");

  int o_ghit;
  int o_gid[MAXHIT];
  double o_ge[MAXHIT];
  double o_gt[MAXHIT];
  int o_tw_ok[MAXHIT];

  tout->Branch("ghit", &o_ghit, "ghit/I");
  tout->Branch("gid", o_gid, "gid[ghit]/I");
  tout->Branch("ge", o_ge, "ge[ghit]/D");
  tout->Branch("gt", o_gt, "gt[ghit]/D");
  tout->Branch("gt_raw", gt_in, "gt_raw[ghit]/D"); // 保留修正前的时间，便于复核
  tout->Branch("tw_ok", o_tw_ok, "tw_ok[ghit]/I");

  Long64_t nentries = tin->GetEntries();
  for (Long64_t ientry = 0; ientry < nentries; ientry++) {

    tin->GetEntry(ientry);

    o_ghit = ghit;
    for (int i = 0; i < ghit; i++) {

      int clusterID = gid_in[i] / 7;
      int segID = gid_in[i] % 7;

      double corr = GetTimeWalkCorrection(clusterID, segID, ge_in[i]);

      o_gid[i] = gid_in[i];
      o_ge[i] = ge_in[i];
      o_gt[i] = gt_in[i] - corr;
      o_tw_ok[i] = clusterID>=0 && clusterID<NCLUSTER && segID>=0 && segID<NSEG
                && has_fit[clusterID][segID] && ge_in[i]>0;
    }

    tout->Fill();
  }

  fout->cd();
  tout->Write();
  for (int c=0; c<NCLUSTER; ++c)
    for (int s=0; s<NSEG; ++s)
      if (has_fit[c][s] && f_tw[c][s]) f_tw[c][s]->Write();
  fout->Close();
  fin->Close();

  std::cout << "time-corrected file saved to "
            << outputFile << std::endl;
}


<h3 id="%E5%90%8C%E4%B8%80Cluster%E5%86%85%E4%B8%8D%E5%90%8Cid%E4%B9%8B%E9%97%B4%E7%9A%84toff">同一Cluster内不同id之间的toff</h3><ul>
<li>可以看出几何上相邻单元之间才有明显的关联。</li>
</ul>


In [12]:
make_pair_timewalk_cluster(6);
draw_pair_offset_cluster(6);

cluster 6 pair time-walk histograms filled.
segment 0 high-energy count = 38109, walk entries = 7236
segment 1 high-energy count = 36768, walk entries = 7083
segment 2 high-energy count = 37226, walk entries = 7120
segment 3 high-energy count = 39352, walk entries = 7345
segment 4 high-energy count = 38709, walk entries = 7518
segment 5 high-energy count = 36949, walk entries = 7314
segment 6 high-energy count = 42415, walk entries = 15350



<h3 id="%E5%90%88%E5%B9%B6%E5%90%8E%E5%90%8C%E4%B8%80Cluster%E5%86%85%E4%B8%8D%E5%90%8Cid%E4%B9%8B%E7%9A%84timewalk">合并后同一Cluster内不同id之的timewalk</h3>


In [13]:
draw_pair_timewalk_cluster(6);


<h3 id="timewalk%E6%8B%9F%E5%90%88">timewalk拟合</h3>
<p>函数参数初值为 <code>(0,1000,−10000,100000)</code>，用于启动数值优化；能量单位 keV、时间单位 ns。拟合范围为 30–800 keV，画出的延伸段不是额外测量。先检查拟合状态，再在 600 keV 处归一化并加入相对参考 segment 的常数 offset。</p>

In [14]:
fit_pair_timewalk_cluster(6, true);

cluster 6, segment 0 fit: p0=39.4166, p1=-646.399, p2=-11067.2, p3=207189
cluster 6, segment 1 fit: p0=51.1366, p1=-988.767, p2=-7951.39, p3=152485
cluster 6, segment 2 fit: p0=38.9093, p1=-560.6, p2=-12613.7, p3=268957
cluster 6, segment 3 fit: p0=48.5325, p1=-1039.83, p2=-7714.07, p3=132969
cluster 6, segment 4 fit: p0=53.0691, p1=-1076.93, p2=-6734, p3=129831
cluster 6, segment 5 fit: p0=59.4729, p1=-1265.45, p2=-5295.2, p3=130088
cluster 6, segment 6 fit: p0=32.7498, p1=-385.362, p2=-13004.3, p3=239117
cluster 6, reference candidate segment 0: high-energy count = 38109
cluster 6, reference candidate segment 1: high-energy count = 36768
cluster 6, reference candidate segment 2: high-energy count = 37226
cluster 6, reference candidate segment 3: high-energy count = 39352
cluster 6, reference candidate segment 4: high-energy count = 38709
cluster 6, reference candidate segment 5: high-energy count = 36949
cluster 6, reference candidate segment 6: high-energy count = 42415
cluster 6: s


<h3 id="%E5%B0%86timewalk%E4%BF%AE%E6%AD%A3%E5%86%99%E5%85%A5root%E6%96%87%E4%BB%B6">将timewalk修正写入root文件</h3>
<p>输出保留 <code>ghit/gid/ge</code> 和各事件的 hit 对应关系；<code>gt</code> 为修正后的时间，<code>gt_raw</code> 保留原时间，<code>tw_ok</code> 表示该通道是否有有效修正。有效拟合函数一并写入文件。这里不做跨 cluster 常数 offset 的二次校准。</p><p>参考传递使用 $C_s=t_{\rm off,sj}+C_j$；没有直接 pair 时通过已知通道连接，不因缺少单个 pair 就放弃该晶体的修正。可用其他有效 pair 检查闭合差。</p>

In [15]:
fit_all_clusters_pair();
make_time_corrected_tree_pair("eurica_event.root", "eurica_time_pair.root");

Processing cluster 0
cluster 0 pair time-walk histograms filled.
segment 0 high-energy count = 23477, walk entries = 4070
segment 1 high-energy count = 21592, walk entries = 4094
segment 2 high-energy count = 24359, walk entries = 4604
segment 3 high-energy count = 27349, walk entries = 5046
segment 4 high-energy count = 27472, walk entries = 4820
segment 5 high-energy count = 23838, walk entries = 4330
segment 6 high-energy count = 25520, walk entries = 9093
cluster 0, segment 0 fit: p0=46.9143, p1=-908.998, p2=-9770.05, p3=199874
cluster 0, segment 1 fit: p0=39.1479, p1=-545.049, p2=-11566.5, p3=220851
cluster 0, segment 2 fit: p0=18.8904, p1=5.97335, p2=-16258.6, p3=285243
cluster 0, segment 3 fit: p0=17.6447, p1=174.833, p2=-17614.4, p3=319376
cluster 0, segment 4 fit: p0=31.1526, p1=-209.942, p2=-16754.1, p3=328047
cluster 0, segment 5 fit: p0=61.401, p1=-1571.08, p2=-814.486, p3=29498.1
cluster 0, segment 6 fit: p0=48.6658, p1=-1018.22, p2=-6539.74, p3=111728
cluster 0, reference